# ドローン観測計画を「単位時間ごとに再最適化」する (Fixstars Amplify)

新緑県の 8 市町村を、センサーを積んだ 5 機のドローンで 10 ステップかけて観測します。
充電ステーションでは観測できない代わりにバッテリーが回復します。

全 10 ステップを 1 つの QUBO にすると大きすぎて時間がかかるので、
**先読み `horizon` ステップだけを解き、先頭の `unit` ステップ（単位時間）だけを実行して、また解き直す**
（ローリングホライズン）方式にしています。

# インストール

In [ ]:
!pip install amplify

アクセストークンを設定

In [ ]:
from getpass import getpass
token = getpass('アクセストークンを貼り付けてEnterキーを押して下さい ')

# ファイルを用意

GitHub からコードとデータを取ってきます（アップロード不要）。

In [ ]:
!git clone -q -b claude/request-5v4fsc https://github.com/mori560269/Q-Questgrow-.git
%cd Q-Questgrow-
!apt-get -qq install -y fonts-ipafont-gothic > /dev/null  # 動画の日本語表示用

モジュールをインポート

In [ ]:
import matplotlib.font_manager as fm
try:  # 動画の日本語表示用フォントを登録
    fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf')
except Exception:
    pass
import numpy as np
import geo_common as g
from rolling_geo import State
from rolling_geo_amplify import build_model, run
from amplify import AmplifyAEClient, solve
from datetime import timedelta

# 1 つの窓のモデルを見てみる

`build_model(S, st, H)` は、現在の状態 `st`（位置・バッテリー・観測済みセル・alpha/beta の累計）から
`H` ステップぶんのモデルを作ります。中身はハンズオンと同じ書き方です。

- 変数: `x = gen.array("Binary", shape=(D, H, C))`（機体 × ステップ × セル）
- コスト関数: 被覆の報酬、観測の重複ペナルティ、alpha/beta の不足（整数スラック付き 2 乗）
- 罰金項: 移動できないセルへの移動
- 制約: `one_hot(x, axis=2)`（各機・各ステップで 1 か所）、`greater_equal(...)`（バッテリーが 0 未満にならない）

In [ ]:
S = g.build_scenario()
st = State(S)
model, x = build_model(S, st, H=3)
print('変数の形', x.shape, ' 制約の数', len(model.constraints))

SQA を実行（1 窓だけ）

In [ ]:
client = AmplifyAEClient()
client.token = token
client.parameters.time_limit_ms = timedelta(milliseconds=1000)  # タイムアウト 1000 ミリ秒

result = solve(model, client, filter_solution=False)
sample_array = x.evaluate(result.best.values)
print('各機・各ステップの位置 (セル番号)')
print(sample_array.argmax(axis=2))

# 単位時間ごとに再最適化して 10 ステップぶん計画する

- `unit`: 1 回の最適化で確定するステップ数（単位時間）
- `horizon`: 先読みするステップ数

各窓では、返ってきた解を厳密な評価関数で採点し、制約違反があれば修復してから、いちばん良い解を採用します。

In [ ]:
res, path, S = run(unit=2, horizon=3, backend='amplify', token=token, time_limit_ms=1000)
print('score', round(res['score'], 3), ' 要求達成', res['requirements_met'], '/', res['requirements_total'],
      ' 合計時間', round(res['total_sec'], 1), '秒')

動画にする

In [ ]:
from IPython.display import Video
g.make_video(S, path, 'rolling_amplify.mp4', title='ローリング最適化 (Fixstars Amplify AE)',
             subtitle=f"score={res['score']:.3f} / 要求達成 {res['requirements_met']}/{res['requirements_total']}")
Video('rolling_amplify.mp4', embed=True, width=900)

# 記録を見る

窓（再最適化 1 回）ごとの記録です。
`raw_feasible` は、ソルバーの解のうち修復なしで制約を満たしていた数です。

In [ ]:
import pandas as pd
pd.DataFrame(res['windows'])

市町村ごとの達成状況です（被覆率・alpha 合計・beta 合計と、それぞれの要求値）。

In [ ]:
pd.DataFrame({'市町村': [m['name'] for m in S.muni],
              '被覆率': res['cov'], '要求被覆率': S.req_cov,
              'alpha': res['A'], '要求alpha': S.req_a,
              'beta': res['B'], '要求beta': S.req_b})

リポジトリに保存してある他の手法の結果と並べて比べます。

In [ ]:
import json, glob
rows = []
for f in sorted(glob.glob('results/result_*geo*.json')):
    r = json.load(open(f))
    rows.append(dict(ファイル=f, 手法=r['method'], score=round(r['score'], 3),
                     要求達成=f"{r['requirements_met']}/{r['requirements_total']}",
                     時間_秒=round(r.get('total_sec', r.get('solve_sec', 0)), 1)))
rows.append(dict(ファイル='(今回)', 手法=res['method'], score=round(res['score'], 3),
                 要求達成=f"{res['requirements_met']}/{res['requirements_total']}",
                 時間_秒=round(res['total_sec'], 1)))
pd.DataFrame(rows)

結果と動画を手元に保存します。

In [ ]:
from google.colab import files
json.dump(res, open('result_rolling_geo_amplify_ae.json', 'w'), ensure_ascii=False, indent=1)
files.download('result_rolling_geo_amplify_ae.json')
files.download('rolling_amplify.mp4')

# 単位時間と先読みを変えて比べる

窓の数 × `time_limit_ms` がおおよその求解時間です。

In [ ]:
for unit, horizon in [(1, 2), (1, 3), (2, 3), (2, 4)]:
    r, _, _ = run(unit=unit, horizon=horizon, backend='amplify', token=token,
                  time_limit_ms=1000, verbose=False)
    print(f"unit={unit} horizon={horizon}  score={r['score']:.3f}  "
          f"要求達成 {r['requirements_met']}/{r['requirements_total']}  合計 {r['total_sec']:.1f}秒")